In [1]:
print("hello world")

hello world


In [3]:
import ollama
import chromadb
import json

# 1. Initialize ChromaDB Client and Collection
# In production, replace the in-memory client with a persistent path
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="api_metadata_store")

# Clear existing data for fresh indexing
existing_ids = collection.get()["ids"]
if existing_ids:
    collection.delete(ids=existing_ids)


# 2. Define sample API metadata database
api_database = [
    {
        "api_id": "api_1",
        "title": "Stripe API",
        "description": "Process credit cards, manage subscriptions, and handle global payouts.",
        "category": "Payments",
        "tags": ["payment", "billing", "card", "fintech"]
    },
    {
        "api_id": "api_2",
        "title": "SendGrid API",
        "description": "cooking country", #"Reliable transactional email delivery, marketing campaigns, and parsing.",
        "category": "Account", #"Communication",
        #"tags": ["email", "transactional", "marketing", "SMTP"]
        "tags": ["india", "america", "africa", "cooking"]
    },
    {
        "api_id": "api_3",
        "title": "OpenWeatherMap API",
        "description": "Access current weather data, historical forecasts, and climate predictions globally.",
        "category": "Weather",
        "tags": ["weather", "forecast", "climate", "location"]
    }
]

# 3. Choose embedding model
# Make sure to run `ollama pull nomic-embed-text` in your terminal first
EMBEDDING_MODEL = "nomic-embed-text"

def get_embedding(text: str):
    """Generate vector embedding using local Ollama model."""
    response = ollama.embeddings(model=EMBEDDING_MODEL, prompt=text)
    print(" created embedding for text:", text)
    return response["embedding"]

# 4. Create Embeddings and Store in Vector DB
print("Indexing APIs...")
for api in api_database:
    # Concatenate the fields that hold semantic meaning for the search
    semantic_text = f"{api['title']} {api['description']} {' '.join(api['tags'])}"
    
    # Generate embedding
    vector = get_embedding(semantic_text)
    
    # Store in ChromaDB
    collection.add(
        documents=[semantic_text],
        metadatas=[{"api_id": api["api_id"], "title": api["title"], "category": api["category"]}],
        ids=[api["api_id"]],
        embeddings=[vector]
    )
print("Indexing complete.")

# 5. Retrieve from user query
def search_api(user_query: str, top_k: int = 1):
    """Search for the best matching API using semantic search."""
    print(f"\nSearching for: '{user_query}'")
    
    # Convert query to vector
    query_vector = get_embedding(user_query)
    
    # Query ChromaDB for nearest neighbor
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k
    )
    
    # Display results
    if results["metadatas"]:
        for i, match in enumerate(results["metadatas"][0]):
            print(f"\nMatch {i+1}:")
            print(f"Title: {match['title']}")
            print(f"Category: {match['category']}")
            print(f"Details: {results['documents'][0][i]}")
    else:
        print("No matching APIs found.")

# 6. Example Query
search_api("I need to send marketing and transactional emails")
#search_api("How can I accept credit card payments on my website?")


Indexing APIs...
 created embedding for text: Stripe API Process credit cards, manage subscriptions, and handle global payouts. payment billing card fintech
 created embedding for text: SendGrid API cooking country india america africa cooking
 created embedding for text: OpenWeatherMap API Access current weather data, historical forecasts, and climate predictions globally. weather forecast climate location
Indexing complete.

Searching for: 'I need to send marketing and transactional emails'
 created embedding for text: I need to send marketing and transactional emails

Match 1:
Title: Stripe API
Category: Payments
Details: Stripe API Process credit cards, manage subscriptions, and handle global payouts. payment billing card fintech


In [4]:
import os
import json
import ollama
import chromadb
from typing import List, Dict

# Configuration Constants
JSON_FILE_PATH = "apismetadata.json"
EMBEDDING_MODEL = "nomic-embed-text"
COLLECTION_NAME = "api_metadata_store"

# 1. Initialize In-Memory Vector DB Client
chroma_client = chromadb.Client()

# Create or reset collection
try:
    chroma_client.delete_collection(name=COLLECTION_NAME)
except Exception:
    pass
collection = chroma_client.create_collection(name=COLLECTION_NAME)


# 2. Ingest: Load JSON File into Array of API Metadata
def ingest_api_metadata(file_path: str) -> List[Dict]:
    """Loads and validates the API metadata from a JSON file."""
    if not os.path.exists(file_path):
        print(f"Error: {file_path} not found. Please create the file first.")
        return []
        
    with open(file_path, "r", encoding="utf-8") as f:
        try:
            api_array = json.load(f)
            print(f"Successfully ingested {len(api_array)} APIs from {file_path}.")
            return api_array
        except json.JSONDecodeError as e:
            print(f"Failed to parse JSON file: {e}")
            return []


# 3. Embedding Pipeline: Process and Vectorize metadata
def get_embedding(text: str) -> List[float]:
    """Generate vector embedding using local Ollama model."""
    response = ollama.embeddings(model=EMBEDDING_MODEL, prompt=text)
    return response["embedding"]


def index_api_database(api_list: List[Dict]):
    """Iterates through API metadata, builds search contexts, embeds, and stores them."""
    if not api_list:
        print("No metadata available to index.")
        return

    print(f"Generating vectors using '{EMBEDDING_MODEL}'...")
    
    for api in api_list:
        # Build dense text block for semantic analysis
        semantic_text = f"Title: {api['title']}. Description: {api['description']}. Keywords: {' '.join(api['tags'])}"
        
        # Pass to the embedding pipeline
        vector = get_embedding(semantic_text)
        
        # Save to Vector Store
        collection.add(
            documents=[semantic_text],
            metadatas=[{
                "api_id": api["api_id"], 
                "title": api["title"], 
                "category": api["category"]
            }],
            ids=[api["api_id"]],
            embeddings=[vector]
        )
    print("Vector database indexing complete.")


# 4. Semantic Search Interface
def search_api(user_query: str, top_k: int = 1):
    """Searches for the closest API vector match using cosine/distance similarity."""
    print(f"\n[Search Query]: '{user_query}'")
    
    # Vectorize the user's natural language question
    query_vector = get_embedding(user_query)
    
    # Query database
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k
    )
    
    # Display the results neatly
    if results and results["metadatas"] and results["metadatas"][0]:
        for i in range(len(results["metadatas"][0])):
            metadata = results["metadatas"][0][i]
            document = results["documents"][0][i]
            print(f"-> Top Match Found: {metadata['title']} ({metadata['category']})")
            print(f"   Indexed Text: {document}")
    else:
        print("-> No matches found.")


# --- Execution Flow ---
if __name__ == "__main__":
    # Step 1: Run Ingestion
    loaded_apis = ingest_api_metadata(JSON_FILE_PATH)
    
    # Step 2: Feed data to embedding system
    index_api_database(loaded_apis)
    
    # Step 3: Test natural language semantic retrieval
    search_api("How can my application send phone verification codes via SMS?", top_k=1)
    search_api("I need a reliable tool to handle credit card billing cycles.", top_k=1)


Successfully ingested 4 APIs from apismetadata.json.
Generating vectors using 'nomic-embed-text'...
Vector database indexing complete.

[Search Query]: 'How can my application send phone verification codes via SMS?'
-> Top Match Found: Twilio SMS & Voice API (Communication)
   Indexed Text: Title: Twilio SMS & Voice API. Description: Programmatically send text messages, initiate phone calls, and build automated verification systems.. Keywords: sms voice otp phone text

[Search Query]: 'I need a reliable tool to handle credit card billing cycles.'
-> Top Match Found: Stripe Payment Gateway (Finance)
   Indexed Text: Title: Stripe Payment Gateway. Description: Accept global payments, process credit cards, manage recurring billing, and track payouts securely.. Keywords: payment billing credit-card subscription checkout


In [5]:
# with schemas

import os
import json
import ollama
import chromadb
from typing import List, Dict

# Configuration Constants
JSON_FILE_PATH = "apisschemas.json"
EMBEDDING_MODEL = "nomic-embed-text"
COLLECTION_NAME = "api_schema_store"

# Initialize ChromaDB Client
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection(name=COLLECTION_NAME)
except Exception:
    pass
collection = chroma_client.create_collection(name=COLLECTION_NAME)


# 1. Ingest JSON metadata
def ingest_api_metadata(file_path: str) -> List[Dict]:
    """Loads and validates the API metadata from a JSON file."""
    if not os.path.exists(file_path):
        print(f"Error: {file_path} not found.")
        return []
    with open(file_path, "r", encoding="utf-8") as f:
        try:
            return json.load(f)
        except json.JSONDecodeError as e:
            print(f"Failed to parse JSON: {e}")
            return []


# 2. Schema Parser: Convert JSON schemas into semantic text
def flatten_schema_to_text(schema_dict: Dict, schema_type: str) -> str:
    """Converts a request/response schema dictionary into clean semantic text sentences."""
    if not schema_dict:
        return f"No explicit {schema_type} parameters."
    
    sentences = [f"The {schema_type} payload contains the following fields:"]
    for field, field_desc in schema_dict.items():
        sentences.append(f"field '{field}' which accepts data type or format '{field_desc}'.")
    
    return " ".join(sentences)


# 3. Embedding Generator
def get_embedding(text: str) -> List[float]:
    """Generate vector embedding using local Ollama model."""
    response = ollama.embeddings(model=EMBEDDING_MODEL, prompt=text)
    return response["embedding"]


# 4. Pipeline Engine
def index_api_database(api_list: List[Dict]):
    """Flattens metadata, requests, and responses into a single semantic string for vector storage."""
    if not api_list:
        return

    print(f"Indexing metadata with schema context using '{EMBEDDING_MODEL}'...")
    
    for api in api_list:
        # Build standard metadata block
        base_text = f"Title: {api['title']}. Description: {api['description']}. Keywords: {' '.join(api['tags'])}."
        
        # Flatten structural schemas
        request_text = flatten_schema_to_text(api.get("request_schema", {}), "request")
        response_text = flatten_schema_to_text(api.get("response_schema", {}), "response")
        
        # Merge all data into a complete semantic block
        final_semantic_text = f"{base_text} {request_text} {response_text}"
        
        # Process vector embedding
        vector = get_embedding(final_semantic_text)
        
        # Add payload and tracking to vector database
        collection.add(
            documents=[final_semantic_text],
            metadatas=[{
                "api_id": api["api_id"], 
                "title": api["title"], 
                "category": api["category"]
            }],
            ids=[api["api_id"]],
            embeddings=[vector]
        )
    print("Vector database indexing complete.")


# 5. Semantic Search Function
def search_api(user_query: str, top_k: int = 1):
    """Searches for APIs matching natural language queries based on descriptions or structural payloads."""
    print(f"\n[Search Query]: '{user_query}'")
    query_vector = get_embedding(user_query)
    
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k
    )
    
    if results and results["metadatas"] and results["metadatas"][0]:
        for i in range(len(results["metadatas"][0])):
            metadata = results["metadatas"][0][i]
            document = results["documents"][0][i]
            print(f"-> Match: {metadata['title']} ({metadata['category']})")
            print(f"   Full Embedded Text: {document}\n")
    else:
        print("-> No matching APIs found.")


# --- Run Pipeline ---
if __name__ == "__main__":
    # Load raw JSON from disk
    loaded_apis = ingest_api_metadata(JSON_FILE_PATH)
    
    # Process text structures, convert to vectors, and store
    index_api_database(loaded_apis)
    
    # --- Semantic Tests ---
    # Test 1: Finding by request parameters
    search_api("Which API takes an amount parameter in cents and a currency code?")
    
    # Test 2: Finding by response fields
    search_api("I need an endpoint that returns a message_sid or message tracker token status.")


Indexing metadata with schema context using 'nomic-embed-text'...
Vector database indexing complete.

[Search Query]: 'Which API takes an amount parameter in cents and a currency code?'
-> Match: Twilio SMS API (Communication)
   Full Embedded Text: Title: Twilio SMS API. Description: Programmatically send text messages and automated verification codes.. Keywords: sms otp phone. The request payload contains the following fields: field 'to_phone_number' which accepts data type or format 'string (E.164 format international phone number)'. field 'message_body' which accepts data type or format 'string (text content max 160 characters)'. field 'sender_id' which accepts data type or format 'string (optional alphanumeric sender masking)'. The response payload contains the following fields: field 'message_sid' which accepts data type or format 'string (unique message tracking token)'. field 'status' which accepts data type or format 'string (queued, sent, delivered, failed)'. field 'error_cod

In [6]:
#swagger openapi standard json

import os
import json
import ollama
import chromadb
from typing import Dict, List

# Configuration constants
OPENAPI_FILE_PATH = "swaggertest.json"
EMBEDDING_MODEL = "nomic-embed-text"
COLLECTION_NAME = "openapi_spec_store"

# Initialize ChromaDB Vector Database
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection(name=COLLECTION_NAME)
except Exception:
    pass
collection = chroma_client.create_collection(name=COLLECTION_NAME)


def parse_json_schema_properties(schema_obj: Dict) -> str:
    """Recursively processes OpenAPI schema properties to build field descriptions."""
    if not schema_obj or "properties" not in schema_obj:
        return "No specific fields defined."
        
    fields = []
    for prop_name, prop_details in schema_obj["properties"].items():
        prop_type = prop_details.get("type", "unknown")
        prop_desc = prop_details.get("description", "No description provided.")
        fields.append(f"field '{prop_name}' ({prop_type}: {prop_desc})")
        
    return ", ".join(fields)


def flatten_openapi_to_semantic_chunks(openapi_data: Dict) -> List[Dict]:
    """
    Parses a global OpenAPI spec and returns a clean array of structured text chunks
    representing individual HTTP endpoints.
    """
    chunks = []
    api_title = openapi_data.get("info", {}).get("title", "Generic API")
    paths = openapi_data.get("paths", {})
    
    for path, methods in paths.items():
        for method, details in methods.items():
            summary = details.get("summary", "")
            description = details.get("description", "")
            
            # --- Extract Request Body Schema Fields ---
            request_fields_text = ""
            try:
                content_types = details.get("requestBody", {}).get("content", {})
                # Safely fallback to application/json schema lookup
                json_schema = content_types.get("application/json", {}).get("schema", {})
                if json_schema:
                    request_fields_text = parse_json_schema_properties(json_schema)
            except Exception:
                request_fields_text = "No request payload parameters required."

            # --- Extract Response Body Schema Fields (Status 200/201) ---
            response_fields_text = ""
            try:
                responses = details.get("responses", {})
                success_response = responses.get("200") or responses.get("201")
                if success_response:
                    resp_schema = success_response.get("content", {}).get("application/json", {}).get("schema", {})
                    response_fields_text = parse_json_schema_properties(resp_schema)
            except Exception:
                response_fields_text = "Standard empty metadata payload return value."

            # Build semantic paragraph for high-density embedding evaluation
            semantic_text = (
                f"API Service Provider: {api_title}. Endpoint route: {method.upper()} {path}. "
                f"Summary: {summary}. Operational Action Details: {description}. "
                f"Expected Client Input Parameters: {request_fields_text}. "
                f"Returned Schema Parameters: {response_fields_text}."
            )
            
            # Add metadata for structural query mapping
            chunks.append({
                "id": f"{method.upper()}_{path.replace('/', '_')}",
                "text": semantic_text,
                "metadata": {
                    "api_title": api_title,
                    "endpoint": f"{method.upper()} {path}",
                    "summary": summary
                }
            })
            
    return chunks


def get_embedding(text: str) -> List[float]:
    """Generate high-density text embeddings using your local Ollama client."""
    response = ollama.embeddings(model=EMBEDDING_MODEL, prompt=text)
    return response["embedding"]


def index_openapi_specification(file_path: str):
    """Ingests OpenAPI spec files, breaks them into vector nodes, and saves them."""
    if not os.path.exists(file_path):
        print(f"Error: {file_path} not found.")
        return
        
    with open(file_path, "r", encoding="utf-8") as f:
        spec_data = json.load(f)
        
    print("Flattening Swagger/OpenAPI schemas into context strings...")
    semantic_chunks = flatten_openapi_to_semantic_chunks(spec_data)
    
    print(f"Embedding {len(semantic_chunks)} endpoint schemas using '{EMBEDDING_MODEL}'...")
    for chunk in semantic_chunks:
        vector = get_embedding(chunk["text"])
        
        collection.add(
            documents=[chunk["text"]],
            metadatas=[chunk["metadata"]],
            ids=[chunk["id"]],
            embeddings=[vector]
        )
    print("OpenAPI spec indexing completed successfully.")


def search_openapi_catalog(user_query: str):
    """Executes a similarity search against the flattened swagger schema space."""
    print(f"\n[Search Query]: '{user_query}'")
    query_vector = get_embedding(user_query)
    
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=1
    )
    
    if results and results["metadatas"] and results["metadatas"][0]:
        meta = results["metadatas"][0][0]
        doc = results["documents"][0][0]
        print(f"-> Best Match Endpoint: {meta['endpoint']} ({meta['summary']})")
        print(f"   Context String Used: {doc}\n")
    else:
        print("-> No operational match found.")


# --- Engine Pipeline Execution ---
if __name__ == "__main__":
    # Ingest, transform, and store the specs into vector space
    index_openapi_specification(OPENAPI_FILE_PATH)
    
    # Run structural text query validation tests
    search_openapi_catalog("I need a route to send text alerts to customer phone numbers")
    search_openapi_catalog("Find an endpoint that returns a message_id tracking identifier value")


Flattening Swagger/OpenAPI schemas into context strings...
Embedding 1 endpoint schemas using 'nomic-embed-text'...
OpenAPI spec indexing completed successfully.

[Search Query]: 'I need a route to send text alerts to customer phone numbers'
-> Best Match Endpoint: POST /v1/messages/send (Send SMS Message)
   Context String Used: API Service Provider: Core Communication Platform. Endpoint route: POST /v1/messages/send. Summary: Send SMS Message. Operational Action Details: Dispatches an instantaneous text message or verification code to a global phone number.. Expected Client Input Parameters: field 'recipient_phone' (string: E.164 formatted target phone number.), field 'body_text' (string: The textual body content of the sms payload.). Returned Schema Parameters: field 'message_id' (string: Unique alphanumeric delivery tracking identifier.), field 'delivery_status' (string: No description provided.).


[Search Query]: 'Find an endpoint that returns a message_id tracking identifier val

In [2]:
#persistent chromadb

import os
import json
import ollama
import chromadb
from typing import Dict, List

# Configuration constants
OPENAPI_FILE_PATH = "swaggertest.json"
EMBEDDING_MODEL = "nomic-embed-text"
COLLECTION_NAME = "openapi_spec_store"

# --- PERSISTENCE UPDATE ---
# This directory will be automatically created on your machine to save database files.
PERSISTENT_DIR = "./chroma_db_storage"

# Initialize Persistent Storage Client instead of an in-memory client
chroma_client = chromadb.PersistentClient(path=PERSISTENT_DIR)

# Get or create collection. Using 'get_or_create' ensures we don't wipe existing disk files on rerun.
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)

def purge_collection_tables():
    """Fetches and deletes all documents to clean out the database tables."""
    print("Initializing system tables cleanup...")
    
    # Retrieve all IDs currently held within the table collection
    existing_records = collection.get()
    record_ids = existing_records.get("ids", [])
    
    if record_ids:
        print(f"Purging {len(record_ids)} obsolete API vectors from disk...")
        collection.delete(ids=record_ids)
        print("Table cleanup complete. Database is fresh.")
    else:
        print("Database tables are already empty. Ready for ingestion.")


def parse_json_schema_properties(schema_obj: Dict) -> str:
    """Recursively processes OpenAPI schema properties to build field descriptions."""
    if not schema_obj or "properties" not in schema_obj:
        return "No specific fields defined."
        
    fields = []
    for prop_name, prop_details in schema_obj["properties"].items():
        prop_type = prop_details.get("type", "unknown")
        prop_desc = prop_details.get("description", "No description provided.")
        fields.append(f"field '{prop_name}' ({prop_type}: {prop_desc})")
        
    return ", ".join(fields)


def flatten_openapi_to_semantic_chunks(openapi_data: Dict) -> List[Dict]:
    """Parses an OpenAPI spec and returns structured text chunks representing endpoints."""
    chunks = []
    api_title = openapi_data.get("info", {}).get("title", "Generic API")
    paths = openapi_data.get("paths", {})
    
    for path, methods in paths.items():
        for method, details in methods.items():
            summary = details.get("summary", "")
            description = details.get("description", "")
            
            request_fields_text = ""
            try:
                content_types = details.get("requestBody", {}).get("content", {})
                json_schema = content_types.get("application/json", {}).get("schema", {})
                if json_schema:
                    request_fields_text = parse_json_schema_properties(json_schema)
            except Exception:
                request_fields_text = "No request payload parameters required."

            response_fields_text = ""
            try:
                responses = details.get("responses", {})
                success_response = responses.get("200") or responses.get("201")
                if success_response:
                    resp_schema = success_response.get("content", {}).get("application/json", {}).get("schema", {})
                    response_fields_text = parse_json_schema_properties(resp_schema)
            except Exception:
                response_fields_text = "Standard empty metadata payload return value."

            semantic_text = (
                f"API Service Provider: {api_title}. Endpoint route: {method.upper()} {path}. "
                f"Summary: {summary}. Operational Action Details: {description}. "
                f"Expected Client Input Parameters: {request_fields_text}. "
                f"Returned Schema Parameters: {response_fields_text}."
            )
            
            chunks.append({
                "id": f"{method.upper()}_{path.replace('/', '_')}",
                "text": semantic_text,
                "metadata": {
                    "api_title": api_title,
                    "endpoint": f"{method.upper()} {path}",
                    "summary": summary
                }
            })
            
    return chunks


def get_embedding(text: str) -> List[float]:
    """Generate high-density text embeddings using your local Ollama client."""
    response = ollama.embeddings(model=EMBEDDING_MODEL, prompt=text)
    return response["embedding"]


def index_openapi_specification(file_path: str):
    """Ingests OpenAPI spec files, breaks them into vector nodes, and saves them to local disk."""
    # Safety Check: If the collection already contains items on disk, skip indexing to prevent duplicates
    if collection.count() > 0:
        print(f"Database already contains {collection.count()} indexed endpoints. Skipping parsing phase.")
        return

    if not os.path.exists(file_path):
        print(f"Error: {file_path} not found.")
        return
        
    with open(file_path, "r", encoding="utf-8") as f:
        spec_data = json.load(f)
        
    print("Flattening Swagger/OpenAPI schemas into context strings...")
    semantic_chunks = flatten_openapi_to_semantic_chunks(spec_data)
    
    print(f"Embedding {len(semantic_chunks)} endpoint schemas using '{EMBEDDING_MODEL}'...")
    for chunk in semantic_chunks:
        vector = get_embedding(chunk["text"])
        
        collection.add(
            documents=[chunk["text"]],
            metadatas=[chunk["metadata"]],
            ids=[chunk["id"]],
            embeddings=[vector]
        )
    print(f"Vector database indexing complete. Saved to directory: {PERSISTENT_DIR}")


def search_openapi_catalog(user_query: str):
    """Executes a similarity search against the disk-persisted swagger schema space."""
    print(f"\n[Search Query]: '{user_query}'")
    query_vector = get_embedding(user_query)
    
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=1
    )
    
    if results and results["metadatas"] and results["metadatas"][0]:
        meta = results["metadatas"][0][0]
        doc = results["documents"][0][0]
        print(f"-> Best Match Endpoint: {meta['endpoint']} ({meta['summary']})")
        print(f"   Context String Used: {doc}\n")
    else:
        print("-> No operational match found.")


# --- Engine Pipeline Execution ---
if __name__ == "__main__":
    # The first time you run this, it parses openapi.json and writes to disk.
    # Subsequent runs read directly from your local disk database storage.
    purge_collection_tables()
    index_openapi_specification(OPENAPI_FILE_PATH)
    
    # Run structural text query validation tests
    search_openapi_catalog("I need a route to send text alerts to customer phone numbers")


Initializing system tables cleanup...
Purging 1 obsolete API vectors from disk...
Table cleanup complete. Database is fresh.
Flattening Swagger/OpenAPI schemas into context strings...
Embedding 1 endpoint schemas using 'nomic-embed-text'...
Vector database indexing complete. Saved to directory: ./chroma_db_storage

[Search Query]: 'I need a route to send text alerts to customer phone numbers'
-> Best Match Endpoint: POST /v1/messages/send (Send SMS Message)
   Context String Used: API Service Provider: Core Communication Platform. Endpoint route: POST /v1/messages/send. Summary: Send SMS Message. Operational Action Details: Dispatches an instantaneous text message or verification code to a global phone number.. Expected Client Input Parameters: field 'recipient_phone' (string: E.164 formatted target phone number.), field 'body_text' (string: The textual body content of the sms payload.). Returned Schema Parameters: field 'message_id' (string: Unique alphanumeric delivery tracking ident